# Managing Agent State - Saving and Loading Agents

In [1]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
assistant = AssistantAgent(
    name = 'Assistant',
    model_client = model_client,
    system_message = 'You are a helpful assistant who can answer questions and provide information on a wide range of topics. You are friendly, informative, and concise.'
)

response = await assistant.on_messages(
    [TextMessage(content = 'Can you help me with a travel itinerary of France from May to June? Keep it under 30 words.',
    source = 'user')],
    cancellation_token = CancellationToken(),
)

print(response.chat_message)

id='6a5b4f0e-783e-4d19-a1ad-ff31f0417d31' source='Assistant' models_usage=RequestUsage(prompt_tokens=62, completion_tokens=38) metadata={} created_at=datetime.datetime(2026, 6, 27, 3, 55, 58, 930175, tzinfo=datetime.timezone.utc) content='Paris sights, Loire Valley castles, Bordeaux wine tasting, Cannes beaches, Nice promenade, Lavender in Provence, Mont Saint-Michel escape, and end in Strasbourg for Alsatian charm. Enjoy!' type='TextMessage'


In [3]:
agent_state = await assistant.save_state()

In [4]:
print(agent_state)

{'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'Can you help me with a travel itinerary of France from May to June? Keep it under 30 words.', 'source': 'user', 'type': 'UserMessage'}, {'content': 'Paris sights, Loire Valley castles, Bordeaux wine tasting, Cannes beaches, Nice promenade, Lavender in Provence, Mont Saint-Michel escape, and end in Strasbourg for Alsatian charm. Enjoy!', 'thought': None, 'source': 'Assistant', 'type': 'AssistantMessage'}]}}


In [5]:
new_assistant_agent = AssistantAgent(
    name = 'Assistant_agent_2',
    model_client = model_client,
    system_message = 'You are a helpful assistant.'
)

In [6]:
await new_assistant_agent.load_state(agent_state)

In [7]:

response = await new_assistant_agent.on_messages(
    [TextMessage(content = 'Can you tell me what was the last discussion about?',
    source = 'user')],
    cancellation_token = CancellationToken(),
)
print(response.chat_message)

id='1770a7ed-76c3-49a8-81f7-d39fc30a4a02' source='Assistant_agent_2' models_usage=RequestUsage(prompt_tokens=98, completion_tokens=43) metadata={} created_at=datetime.datetime(2026, 6, 27, 3, 56, 10, 806847, tzinfo=datetime.timezone.utc) content='You asked for a travel itinerary for France from May to June and I provided a brief suggestion including destinations like Paris, the Loire Valley, Bordeaux, Cannes, Nice, Provence, Mont Saint-Michel, and Strasbourg.' type='TextMessage'


In [8]:
response.chat_message.content

'You asked for a travel itinerary for France from May to June and I provided a brief suggestion including destinations like Paris, the Loire Valley, Bordeaux, Cannes, Nice, Provence, Mont Saint-Michel, and Strasbourg.'